# PhysicalAI — Colab GPU 백엔드 (ngrok 터널)

배포된 대시보드(`https://tubular-torte-1df9ea.netlify.app`)의 **Run/Training을 Colab GPU에서 실제 실행**하도록 연결합니다.

- `configs/env.yaml`이 `mock_mode: true`라 Isaac Sim 없이 **numpy mock 환경 + torch(CUDA)** 로 IL/RL 학습이 GPU에서 돌아갑니다.
- 백엔드(`uvicorn api.main:app`)를 띄우고 **ngrok 터널**로 공개 HTTPS URL을 만든 뒤, 대시보드에 붙일 주소를 출력합니다.
- **ngrok을 쓰는 이유**: cloudflared 임시 터널은 SSE(Server-Sent Events) 스트림을 버퍼링해서 Live Log·Reward/Loss 차트가 실시간으로 안 흐릅니다. ngrok은 SSE를 그대로 통과시킵니다.

**실행 전 필수:**
1. 런타임 → 런타임 유형 변경 → **T4 GPU**(또는 L4) 선택
2. 무료 ngrok authtoken 준비: <https://dashboard.ngrok.com/get-started/your-authtoken> (가입 후 토큰 복사)

> 주의: Colab 세션/터널 URL은 **임시**입니다(수 시간 후 만료, 재시작 시 URL 변경). 데모·실험용입니다.

## Step 1 — GPU 확인

In [ ]:
import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else "❌ nvidia-smi 실패 — 런타임 유형을 T4 GPU로 변경하세요!")

## Step 2 — 레포 클론 + 의존성 설치

레포의 `demos/ checkpoints/ outputs/`는 Windows 전용 심볼릭 링크라 Colab에선 깨져 있습니다. 제거하고 실제 디렉터리로 다시 만듭니다 (비워둔 상태 → COLLECT가 `episode_0000`부터 생성).

In [ ]:
import subprocess, sys, os, pathlib

REPO = "/content/PhysicalAI"
if not os.path.isdir(REPO):
    print("[1/3] Cloning repo...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/00saridon/PhysicalAI.git", REPO], check=True)
os.chdir(REPO)

for d in ["demos", "checkpoints", "outputs"]:
    p = pathlib.Path(d)
    if p.is_symlink() or p.exists():
        subprocess.run(["rm", "-rf", d])
for d in ["demos", "checkpoints/il", "checkpoints/rl", "outputs/policy", "outputs/dataset"]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print("  dirs ready:", os.listdir(".")[:8], "...")

print("[2/3] Installing deps (torch is already CUDA-ready on Colab)...")
# requirements-api.txt = backend + full pipeline stack (fastapi/uvicorn/sse +
# torch/SB3/onnx/gymnasium). torch>=2.1 is already satisfied by Colab's CUDA build.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-api.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)

print("[3/3] Verifying imports...")
import torch
print("  torch", torch.__version__, "| cuda", torch.cuda.is_available())
import importlib.util
print("  stable_baselines3:", importlib.util.find_spec("stable_baselines3") is not None)
print("✅ Step 2 complete")

## Step 3 — 백엔드 + ngrok 터널 기동

아래 `NGROK_AUTHTOKEN`에 본인 토큰을 붙여넣고 실행하세요 (무료: <https://dashboard.ngrok.com/get-started/your-authtoken>).

`MOCK_PIPELINE=true`로 시작하므로 백엔드는 MOCK 상태로 뜨고, 대시보드의 토글로 **REAL_MODE**(GPU 실학습)로 전환할 수 있습니다 (`real_available=true`).

In [ ]:
NGROK_AUTHTOKEN = ""  # ← 여기에 ngrok authtoken 붙여넣기

import subprocess, sys, os, time, urllib.request, threading

DASHBOARD = "https://tubular-torte-1df9ea.netlify.app"
assert NGROK_AUTHTOKEN.strip(), "NGROK_AUTHTOKEN을 입력하세요 — https://dashboard.ngrok.com/get-started/your-authtoken"
os.chdir("/content/PhysicalAI")

# ── 1) 백엔드 기동 (MOCK으로 시작 — 대시보드에서 REAL로 토글) ──
env = {**os.environ, "MOCK_PIPELINE": "true"}
backend = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/PhysicalAI", env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
threading.Thread(target=lambda: [None for _ in backend.stdout], daemon=True).start()  # drain

ok = False
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:8000/api/health", timeout=2); ok = True; break
    except Exception:
        time.sleep(1)
print("✅ backend up" if ok else "❌ backend failed")

# ── 2) ngrok 터널 (SSE를 버퍼 없이 통과) ──
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
ngrok.kill()  # 기존 터널 정리
tunnel = ngrok.connect(8000, "http")
url = tunnel.public_url.replace("http://", "https://")

print("\n" + "=" * 64)
print("TUNNEL URL :", url)
print("\n👇 이 주소로 대시보드를 열면 백엔드가 이 GPU로 연결됩니다:")
print(f"   {DASHBOARD}/?api={url}")
print("\n(또는 대시보드 Config 페이지의 'Backend (API)'에 위 URL 붙여넣기)")
print("=" * 64)

## Step 4 — 사용 방법

1. 위에서 출력된 `https://<site>/?api=<tunnel>` 링크로 대시보드를 엽니다 (또는 Config → Backend(API)에 터널 URL 입력 후 *적용*).
2. 상단 토글을 **REAL_MODE**로 전환 (이 백엔드는 torch가 있어 활성화됨).
3. **Run** 메뉴에서 순서대로 실행: `ENV → COLLECT → IL → RL → EXPORT`
   - REAL 모드는 선행 산출물 가드가 있으므로 **COLLECT를 먼저** 돌려 `demos/`를 만든 뒤 IL/RL로 진행하세요.
   - RL(PPO 50k steps)이 **GPU**에서 학습됩니다 (CPU 대비 ~100×↑).
4. **Training** 메뉴에서 reward/loss 곡선을 ngrok 터널을 통해 **실시간**으로 확인합니다.

이 노트북 탭은 **열어둔 채로** 두세요 — 닫으면 백엔드와 터널이 종료됩니다.

### 중지
런타임을 종료하거나 아래 셀을 실행하세요.

In [ ]:
# 중지: 터널 + 백엔드 종료
try:
    from pyngrok import ngrok
    ngrok.kill()
    backend.terminate()
    print("stopped backend + ngrok")
except NameError:
    print("nothing to stop (Step 3 을 먼저 실행하세요)")